In [ ]:
import os
PROJECT_ROOT = os.environ.get("MST_PROJECT_ROOT", "..")
from pathlib import Path

BASE_DIR = Path(PROJECT_ROOT)

PYTHON_DIR = BASE_DIR / "03_Python_Optimization"
RESULTS_DIR = BASE_DIR / "05_Results"
FIGURES_DIR = BASE_DIR / "06_Figures"
TABLES_DIR = BASE_DIR / "07_Tables"

# Ensure all output folders exist.
for folder in [PYTHON_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Python:", PYTHON_DIR)
print("Results:", RESULTS_DIR)
print("Figures:", FIGURES_DIR)
print("Tables:", TABLES_DIR)

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class InformationItem:
    """
    A single privacy-admissible information item considered by MST.

    Attributes
    ----------
    item_id : str
        Unique identifier, e.g. "z1".
    name : str
        Human-readable description of the information item.
    utility : float
        Modeled decision-support contribution in [0, 1].
    privacy : float
        Modeled privacy exposure in [0, 1].
    cognitive_cost : float
        Modeled interpretation cost, >= 0.
    """

    item_id: str
    name: str
    utility: float
    privacy: float
    cognitive_cost: float

    def __post_init__(self):
        if not 0.0 <= self.utility <= 1.0:
            raise ValueError(
                f"utility must be in [0, 1], received {self.utility}"
            )

        if not 0.0 <= self.privacy <= 1.0:
            raise ValueError(
                f"privacy must be in [0, 1], received {self.privacy}"
            )

        if self.cognitive_cost < 0.0:
            raise ValueError(
                "cognitive_cost must be non-negative."
            )

In [ ]:
test_item = InformationItem(
    item_id="z1",
    name="Control action taken",
    utility=0.80,
    privacy=0.05,
    cognitive_cost=0.10,
)

print(test_item)

In [ ]:
from collections.abc import Iterable


def mst_utility(items: Iterable[InformationItem]) -> float:
    """
    Bounded cumulative modeled utility:

        U(S) = 1 - product(1 - u(z))

    Returns a value in [0, 1].
    """
    product = 1.0

    for item in items:
        product *= (1.0 - item.utility)

    return 1.0 - product


def mst_privacy(items: Iterable[InformationItem]) -> float:
    """
    Bounded cumulative privacy exposure:

        P(S) = 1 - product(1 - p(z))

    Returns a value in [0, 1].
    """
    product = 1.0

    for item in items:
        product *= (1.0 - item.privacy)

    return 1.0 - product


def mst_cognitive_cost(items: Iterable[InformationItem]) -> float:
    """
    Additive modeled interpretation cost:

        C(S) = sum c(z)
    """
    return sum(item.cognitive_cost for item in items)

In [ ]:
test_items = [
    InformationItem(
        item_id="z1",
        name="Control action taken",
        utility=0.80,
        privacy=0.05,
        cognitive_cost=0.10,
    ),
    InformationItem(
        item_id="z2",
        name="General operational reason",
        utility=0.70,
        privacy=0.05,
        cognitive_cost=0.15,
    ),
    InformationItem(
        item_id="z3",
        name="Current system condition",
        utility=0.50,
        privacy=0.10,
        cognitive_cost=0.20,
    ),
]

In [ ]:
utility_value = mst_utility(test_items)
privacy_value = mst_privacy(test_items)
cognitive_value = mst_cognitive_cost(test_items)

print(f"U(S) = {utility_value:.4f}")
print(f"P(S) = {privacy_value:.4f}")
print(f"C(S) = {cognitive_value:.4f}")

In [ ]:
def mst_disclosure_cost(
    selected_items: Iterable[InformationItem],
    total_items: int,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> float:
    """
    Composite MST disclosure cost:

        J(S) =
            lambda_privacy * P(S)
            + lambda_cognitive * C(S)
            + lambda_cardinality * |S| / |D|

    Parameters
    ----------
    selected_items
        Information items contained in disclosure subset S.

    total_items
        Total number of admissible information items |D|.

    lambda_privacy
        Weight assigned to privacy exposure.

    lambda_cognitive
        Weight assigned to modeled interpretation cost.

    lambda_cardinality
        Small parsimony regularization weight.

    Returns
    -------
    float
        Composite disclosure cost J(S).
    """

    items = list(selected_items)

    if total_items <= 0:
        raise ValueError("total_items must be greater than zero.")

    if any(
        weight < 0
        for weight in (
            lambda_privacy,
            lambda_cognitive,
            lambda_cardinality,
        )
    ):
        raise ValueError("All lambda weights must be non-negative.")

    privacy_cost = mst_privacy(items)
    cognitive_cost = mst_cognitive_cost(items)
    normalized_cardinality = len(items) / total_items

    return (
        lambda_privacy * privacy_cost
        + lambda_cognitive * cognitive_cost
        + lambda_cardinality * normalized_cardinality
    )

In [ ]:
test_disclosure_cost = mst_disclosure_cost(
    selected_items=test_items,
    total_items=len(test_items),
    lambda_privacy=1.0,
    lambda_cognitive=1.0,
    lambda_cardinality=0.01,
)

print(f"J(S) = {test_disclosure_cost:.4f}")

In [ ]:
from itertools import combinations


def evaluate_subset(
    subset: Iterable[InformationItem],
    total_items: int,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Evaluate one disclosure subset using the current MST functions.
    """
    items = list(subset)

    return {
        "items": tuple(item.item_id for item in items),
        "utility": mst_utility(items),
        "privacy": mst_privacy(items),
        "cognitive_cost": mst_cognitive_cost(items),
        "disclosure_cost": mst_disclosure_cost(
            selected_items=items,
            total_items=total_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        ),
    }

In [ ]:
all_subset_results = []

for subset_size in range(len(test_items) + 1):
    for subset in combinations(test_items, subset_size):
        result = evaluate_subset(
            subset=subset,
            total_items=len(test_items),
        )
        all_subset_results.append(result)

for result in all_subset_results:
    print(
        f"{result['items']}: "
        f"U={result['utility']:.4f}, "
        f"P={result['privacy']:.4f}, "
        f"C={result['cognitive_cost']:.4f}, "
        f"J={result['disclosure_cost']:.4f}"
    )

In [ ]:
def exact_mst(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Solve the MST problem exactly by exhaustive subset enumeration.

    Returns the globally minimum-cost subset satisfying U(S) >= tau.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError("tau must be in the interval (0, 1].")

    total_items = len(items)

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    # Under monotone utility, if all admissible information is insufficient,
    # no subset can satisfy the threshold.
    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": mst_utility(items),
            "privacy": mst_privacy(items),
            "cognitive_cost": mst_cognitive_cost(items),
            "disclosure_cost": float("inf"),
        }

    best_result = None

    for subset_size in range(total_items + 1):
        for subset in combinations(items, subset_size):

            result = evaluate_subset(
                subset=subset,
                total_items=total_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )

            if result["utility"] < tau:
                continue

            if best_result is None:
                best_result = result
                continue

            # Primary criterion: minimum composite disclosure cost.
            current_key = (
                result["disclosure_cost"],
                result["privacy"],
                result["cognitive_cost"],
                len(result["items"]),
                result["items"],
            )

            best_key = (
                best_result["disclosure_cost"],
                best_result["privacy"],
                best_result["cognitive_cost"],
                len(best_result["items"]),
                best_result["items"],
            )

            if current_key < best_key:
                best_result = result

    return {
        "status": "OPTIMAL",
        "selected_items": best_result["items"],
        "utility": best_result["utility"],
        "privacy": best_result["privacy"],
        "cognitive_cost": best_result["cognitive_cost"],
        "disclosure_cost": best_result["disclosure_cost"],
    }

In [ ]:
exact_result = exact_mst(
    items=test_items,
    tau=0.90,
)

print("Status:", exact_result["status"])
print("Selected:", exact_result["selected_items"])
print(f"Utility: {exact_result['utility']:.4f}")
print(f"Privacy: {exact_result['privacy']:.4f}")
print(f"Cognitive cost: {exact_result['cognitive_cost']:.4f}")
print(f"Disclosure cost: {exact_result['disclosure_cost']:.4f}")

In [ ]:
test_thresholds = [0.50, 0.75, 0.90, 0.95, 0.98]

for tau in test_thresholds:
    result = exact_mst(
        items=test_items,
        tau=tau,
    )

    print(f"\nTau = {tau:.2f}")
    print("Status:", result["status"])
    print("Selected:", result["selected_items"])
    print(f"Utility: {result['utility']:.4f}")
    print(f"Disclosure cost: {result['disclosure_cost']:.4f}")

In [ ]:
def greedy_mst(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Solve the MST problem using greedy marginal utility per marginal cost.

    At each step, select the remaining item maximizing:

        Delta U(z | S) / (Delta J(z | S) + epsilon)
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError("tau must be in the interval (0, 1].")

    total_items = len(items)

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    # Feasibility check.
    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": mst_utility(items),
            "privacy": mst_privacy(items),
            "cognitive_cost": mst_cognitive_cost(items),
            "disclosure_cost": float("inf"),
        }

    selected = []
    remaining = list(items)

    while mst_utility(selected) < tau:

        best_item = None
        best_score = float("-inf")

        current_utility = mst_utility(selected)
        current_cost = mst_disclosure_cost(
            selected_items=selected,
            total_items=total_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        for item in remaining:

            candidate = selected + [item]

            candidate_utility = mst_utility(candidate)

            candidate_cost = mst_disclosure_cost(
                selected_items=candidate,
                total_items=total_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )

            delta_utility = candidate_utility - current_utility
            delta_cost = candidate_cost - current_cost

            if delta_utility <= 0:
                continue

            score = delta_utility / (delta_cost + epsilon)

            # Deterministic tie-breaking by item ID.
            if (
                score > best_score
                or (
                    abs(score - best_score) <= 1e-12
                    and best_item is not None
                    and item.item_id < best_item.item_id
                )
            ):
                best_item = item
                best_score = score

        if best_item is None:
            return {
                "status": "INFEASIBLE",
                "selected_items": tuple(
                    item.item_id for item in selected
                ),
                "utility": mst_utility(selected),
                "privacy": mst_privacy(selected),
                "cognitive_cost": mst_cognitive_cost(selected),
                "disclosure_cost": float("inf"),
            }

        selected.append(best_item)
        remaining.remove(best_item)

    final_result = evaluate_subset(
        subset=selected,
        total_items=total_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    return {
        "status": "FEASIBLE",
        "selected_items": final_result["items"],
        "utility": final_result["utility"],
        "privacy": final_result["privacy"],
        "cognitive_cost": final_result["cognitive_cost"],
        "disclosure_cost": final_result["disclosure_cost"],
    }

In [ ]:
greedy_result = greedy_mst(
    items=test_items,
    tau=0.90,
)

print("Status:", greedy_result["status"])
print("Selected:", greedy_result["selected_items"])
print(f"Utility: {greedy_result['utility']:.4f}")
print(f"Privacy: {greedy_result['privacy']:.4f}")
print(f"Cognitive cost: {greedy_result['cognitive_cost']:.4f}")
print(f"Disclosure cost: {greedy_result['disclosure_cost']:.4f}")

In [ ]:
comparison_thresholds = [0.50, 0.75, 0.90, 0.95, 0.98]

for tau in comparison_thresholds:

    exact = exact_mst(
        items=test_items,
        tau=tau,
    )

    greedy = greedy_mst(
        items=test_items,
        tau=tau,
    )

    print(f"\nTau = {tau:.2f}")

    print(
        f"Exact  -> Status: {exact['status']}, "
        f"Selected: {exact['selected_items']}, "
        f"J: {exact['disclosure_cost']:.4f}"
    )

    print(
        f"Greedy -> Status: {greedy['status']}, "
        f"Selected: {greedy['selected_items']}, "
        f"J: {greedy['disclosure_cost']:.4f}"
    )

In [ ]:
hard_items = [
    InformationItem(
        item_id="h1",
        name="High immediate utility item",
        utility=0.70,
        privacy=0.00,
        cognitive_cost=0.20,
    ),
    InformationItem(
        item_id="h2",
        name="Moderate utility item A",
        utility=0.60,
        privacy=0.00,
        cognitive_cost=0.18,
    ),
    InformationItem(
        item_id="h3",
        name="Moderate utility item B",
        utility=0.60,
        privacy=0.00,
        cognitive_cost=0.18,
    ),
]

tau_hard = 0.80

exact_hard = exact_mst(
    items=hard_items,
    tau=tau_hard,
)

greedy_hard = greedy_mst(
    items=hard_items,
    tau=tau_hard,
)

print("EXACT-MST")
print("Selected:", exact_hard["selected_items"])
print(f"Utility: {exact_hard['utility']:.4f}")
print(f"Disclosure cost: {exact_hard['disclosure_cost']:.4f}")

print("\nGREEDY-MST")
print("Selected:", greedy_hard["selected_items"])
print(f"Utility: {greedy_hard['utility']:.4f}")
print(f"Disclosure cost: {greedy_hard['disclosure_cost']:.4f}")

In [ ]:
def compute_optimality_gap(
    exact_result: dict,
    approximate_result: dict,
) -> float:
    """
    Compute the relative optimality gap of an approximate MST solution.

    Gap = (J_approx - J_exact) / J_exact

    Returns
    -------
    float
        Relative optimality gap.
        A value of 0 means the approximate solution is optimal.
    """

    if exact_result["status"] != "OPTIMAL":
        raise ValueError(
            "The exact result must contain a feasible optimal solution."
        )

    if approximate_result["status"] not in {"FEASIBLE", "OPTIMAL"}:
        raise ValueError(
            "The approximate result must contain a feasible solution."
        )

    exact_cost = exact_result["disclosure_cost"]
    approximate_cost = approximate_result["disclosure_cost"]

    if exact_cost <= 0:
        if approximate_cost == exact_cost:
            return 0.0

        raise ValueError(
            "Relative optimality gap is undefined when "
            "the exact optimal cost is zero."
        )

    return (
        approximate_cost - exact_cost
    ) / exact_cost

In [ ]:
hard_gap = compute_optimality_gap(
    exact_result=exact_hard,
    approximate_result=greedy_hard,
)

print(f"Relative optimality gap: {hard_gap:.4f}")
print(f"Optimality gap (%): {100 * hard_gap:.2f}%")

In [ ]:
def progressive_mst_step(
    selected_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Perform one Progressive-MST disclosure step.

    Selects the remaining information item with the highest
    marginal utility per marginal disclosure cost.
    """

    total_items = len(all_items)

    selected_ids = {
        item.item_id for item in selected_items
    }

    remaining_items = [
        item
        for item in all_items
        if item.item_id not in selected_ids
    ]

    if not remaining_items:
        return {
            "status": "NO_ADDITIONAL_INFORMATION",
            "selected_items": tuple(
                item.item_id for item in selected_items
            ),
        }

    current_utility = mst_utility(selected_items)

    current_cost = mst_disclosure_cost(
        selected_items=selected_items,
        total_items=total_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    best_item = None
    best_score = float("-inf")

    for item in remaining_items:

        expanded_subset = selected_items + [item]

        new_utility = mst_utility(expanded_subset)

        new_cost = mst_disclosure_cost(
            selected_items=expanded_subset,
            total_items=total_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        delta_utility = new_utility - current_utility
        delta_cost = new_cost - current_cost

        if delta_utility <= 0:
            continue

        score = delta_utility / (delta_cost + epsilon)

        if (
            score > best_score
            or (
                abs(score - best_score) <= 1e-12
                and best_item is not None
                and item.item_id < best_item.item_id
            )
        ):
            best_item = item
            best_score = score

    if best_item is None:
        return {
            "status": "NO_ADDITIONAL_INFORMATION",
            "selected_items": tuple(
                item.item_id for item in selected_items
            ),
        }

    expanded_items = selected_items + [best_item]

    result = evaluate_subset(
        subset=expanded_items,
        total_items=total_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    return {
        "status": "EXPANDED",
        "added_item": best_item.item_id,
        "selected_items": result["items"],
        "utility": result["utility"],
        "privacy": result["privacy"],
        "cognitive_cost": result["cognitive_cost"],
        "disclosure_cost": result["disclosure_cost"],
        "selection_score": best_score,
    }

In [ ]:
initial_disclosure = [test_items[0]]

progressive_result = progressive_mst_step(
    selected_items=initial_disclosure,
    all_items=test_items,
)

print("Status:", progressive_result["status"])
print("Added item:", progressive_result["added_item"])
print("Selected:", progressive_result["selected_items"])
print(f"Utility: {progressive_result['utility']:.4f}")
print(
    f"Disclosure cost: "
    f"{progressive_result['disclosure_cost']:.4f}"
)

In [ ]:
def progressive_mst_sequence(
    initial_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> list[dict]:
    """
    Generate the full Progressive-MST disclosure sequence.
    """

    current_items = list(initial_items)
    history = []

    # Record initial state.
    initial_result = evaluate_subset(
        subset=current_items,
        total_items=len(all_items),
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    history.append(
        {
            "step": 0,
            "added_item": None,
            **initial_result,
        }
    )

    step = 1

    while True:
        result = progressive_mst_step(
            selected_items=current_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        if result["status"] != "EXPANDED":
            break

        added_id = result["added_item"]

        added_item = next(
            item for item in all_items
            if item.item_id == added_id
        )

        current_items.append(added_item)

        history.append(
            {
                "step": step,
                "added_item": added_id,
                "items": result["selected_items"],
                "utility": result["utility"],
                "privacy": result["privacy"],
                "cognitive_cost": result["cognitive_cost"],
                "disclosure_cost": result["disclosure_cost"],
            }
        )

        step += 1

    return history

In [ ]:
progressive_history = progressive_mst_sequence(
    initial_items=[test_items[0]],
    all_items=test_items,
)

for record in progressive_history:
    print(
        f"Step {record['step']}: "
        f"Added={record['added_item']}, "
        f"Selected={record['items']}, "
        f"U={record['utility']:.4f}, "
        f"J={record['disclosure_cost']:.4f}"
    )

In [ ]:
def validate_progressive_properties(history: list[dict]) -> dict:
    """
    Validate theoretical properties of a Progressive-MST sequence.

    Checks:
    1. Nested disclosure.
    2. Non-decreasing modeled utility.
    3. Non-decreasing disclosure cost.
    """

    nested_disclosure = True
    monotonic_utility = True
    monotonic_cost = True

    for previous, current in zip(history, history[1:]):

        previous_items = set(previous["items"])
        current_items = set(current["items"])

        # S^(k) must be a subset of S^(k+1)
        if not previous_items.issubset(current_items):
            nested_disclosure = False

        # U(S^(k)) <= U(S^(k+1))
        if current["utility"] < previous["utility"] - 1e-12:
            monotonic_utility = False

        # J(S^(k)) <= J(S^(k+1))
        if (
            current["disclosure_cost"]
            < previous["disclosure_cost"] - 1e-12
        ):
            monotonic_cost = False

    return {
        "nested_disclosure": nested_disclosure,
        "monotonic_utility": monotonic_utility,
        "monotonic_disclosure_cost": monotonic_cost,
        "all_properties_satisfied": (
            nested_disclosure
            and monotonic_utility
            and monotonic_cost
        ),
    }

In [ ]:
progressive_validation = validate_progressive_properties(
    progressive_history
)

for property_name, passed in progressive_validation.items():
    print(f"{property_name}: {passed}")

In [ ]:
import pandas as pd


# Save the progressive disclosure sequence.
progressive_df = pd.DataFrame(progressive_history)

progressive_output = (
    RESULTS_DIR / "progressive_mst_validation_sequence.csv"
)

progressive_df.to_csv(
    progressive_output,
    index=False,
)


# Save the theoretical property validation results.
validation_df = pd.DataFrame(
    [
        {
            "property": key,
            "passed": value,
        }
        for key, value in progressive_validation.items()
    ]
)

validation_output = (
    RESULTS_DIR / "progressive_mst_property_validation.csv"
)

validation_df.to_csv(
    validation_output,
    index=False,
)


print("Saved successfully:")
print(progressive_output)
print(validation_output)

In [ ]:
comparison_data = [
    {
        "method": "Exact-MST",
        "status": exact_hard["status"],
        "selected_items": exact_hard["selected_items"],
        "utility": exact_hard["utility"],
        "privacy": exact_hard["privacy"],
        "cognitive_cost": exact_hard["cognitive_cost"],
        "disclosure_cost": exact_hard["disclosure_cost"],
        "optimality_gap": 0.0,
        "optimality_gap_percent": 0.0,
    },
    {
        "method": "Greedy-MST",
        "status": greedy_hard["status"],
        "selected_items": greedy_hard["selected_items"],
        "utility": greedy_hard["utility"],
        "privacy": greedy_hard["privacy"],
        "cognitive_cost": greedy_hard["cognitive_cost"],
        "disclosure_cost": greedy_hard["disclosure_cost"],
        "optimality_gap": hard_gap,
        "optimality_gap_percent": 100 * hard_gap,
    },
]

counterexample_df = pd.DataFrame(comparison_data)

counterexample_output = (
    RESULTS_DIR / "exact_vs_greedy_counterexample.csv"
)

counterexample_df.to_csv(
    counterexample_output,
    index=False,
)

print(counterexample_df)
print("\nSaved successfully:")
print(counterexample_output)

In [ ]:
module_path = PYTHON_DIR / "mst_core.py"

module_part_1 = r'''
from __future__ import annotations

from dataclasses import dataclass
from collections.abc import Iterable


@dataclass(frozen=True)
class InformationItem:
    """
    A single privacy-admissible information item considered by MST.
    """

    item_id: str
    name: str
    utility: float
    privacy: float
    cognitive_cost: float

    def __post_init__(self):
        if not 0.0 <= self.utility <= 1.0:
            raise ValueError(
                f"utility must be in [0, 1], received {self.utility}"
            )

        if not 0.0 <= self.privacy <= 1.0:
            raise ValueError(
                f"privacy must be in [0, 1], received {self.privacy}"
            )

        if self.cognitive_cost < 0.0:
            raise ValueError(
                "cognitive_cost must be non-negative."
            )


def mst_utility(
    items: Iterable[InformationItem],
) -> float:
    """
    Bounded cumulative modeled decision-support utility.

    U(S) = 1 - product(1 - u(z))
    """

    product = 1.0

    for item in items:
        product *= 1.0 - item.utility

    return 1.0 - product


def mst_privacy(
    items: Iterable[InformationItem],
) -> float:
    """
    Bounded cumulative privacy exposure.

    P(S) = 1 - product(1 - p(z))
    """

    product = 1.0

    for item in items:
        product *= 1.0 - item.privacy

    return 1.0 - product


def mst_cognitive_cost(
    items: Iterable[InformationItem],
) -> float:
    """
    Additive modeled interpretation cost.

    C(S) = sum c(z)
    """

    return sum(
        item.cognitive_cost
        for item in items
    )
'''

module_path.write_text(
    module_part_1.strip() + "\n",
    encoding="utf-8",
)

print("Created:")
print(module_path)

In [ ]:
module_part_2 = r'''

def mst_disclosure_cost(
    selected_items: Iterable[InformationItem],
    total_items: int,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> float:
    """
    Composite MST disclosure cost.

    J(S) =
        lambda_privacy * P(S)
        + lambda_cognitive * C(S)
        + lambda_cardinality * |S| / |D|
    """

    items = list(selected_items)

    if total_items <= 0:
        raise ValueError(
            "total_items must be greater than zero."
        )

    if any(
        weight < 0
        for weight in (
            lambda_privacy,
            lambda_cognitive,
            lambda_cardinality,
        )
    ):
        raise ValueError(
            "All lambda weights must be non-negative."
        )

    privacy_cost = mst_privacy(items)
    cognitive_cost = mst_cognitive_cost(items)

    normalized_cardinality = (
        len(items) / total_items
    )

    return (
        lambda_privacy * privacy_cost
        + lambda_cognitive * cognitive_cost
        + lambda_cardinality * normalized_cardinality
    )


def evaluate_subset(
    subset: Iterable[InformationItem],
    total_items: int,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Evaluate one MST disclosure subset.
    """

    items = list(subset)

    return {
        "items": tuple(
            item.item_id
            for item in items
        ),
        "utility": mst_utility(items),
        "privacy": mst_privacy(items),
        "cognitive_cost": mst_cognitive_cost(items),
        "disclosure_cost": mst_disclosure_cost(
            selected_items=items,
            total_items=total_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        ),
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_2)

print("Appended disclosure-cost functions to:")
print(module_path)

In [ ]:
import sys

python_folder = str(PYTHON_DIR)

if python_folder not in sys.path:
    sys.path.insert(0, python_folder)

print("Module search path added:")
print(python_folder)

In [ ]:
import importlib
import mst_core

importlib.reload(mst_core)

print("mst_core imported successfully.")
print(mst_core.__file__)

In [ ]:
module_part_3 = r'''

from itertools import combinations


def exact_mst(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Solve the MST problem exactly by exhaustive subset enumeration.

    Returns the globally minimum-cost subset satisfying U(S) >= tau.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError(
            "tau must be in the interval (0, 1]."
        )

    total_items = len(items)

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": mst_utility(items),
            "privacy": mst_privacy(items),
            "cognitive_cost": mst_cognitive_cost(items),
            "disclosure_cost": float("inf"),
        }

    best_result = None

    for subset_size in range(total_items + 1):

        for subset in combinations(
            items,
            subset_size,
        ):

            result = evaluate_subset(
                subset=subset,
                total_items=total_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )

            if result["utility"] < tau:
                continue

            if best_result is None:
                best_result = result
                continue

            current_key = (
                result["disclosure_cost"],
                result["privacy"],
                result["cognitive_cost"],
                len(result["items"]),
                result["items"],
            )

            best_key = (
                best_result["disclosure_cost"],
                best_result["privacy"],
                best_result["cognitive_cost"],
                len(best_result["items"]),
                best_result["items"],
            )

            if current_key < best_key:
                best_result = result

    return {
        "status": "OPTIMAL",
        "selected_items": best_result["items"],
        "utility": best_result["utility"],
        "privacy": best_result["privacy"],
        "cognitive_cost": best_result["cognitive_cost"],
        "disclosure_cost": best_result["disclosure_cost"],
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_3)

print("Exact-MST added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available Exact-MST function:")
print(mst_core.exact_mst)

In [ ]:
module_test_items = [
    mst_core.InformationItem(
        item_id="z1",
        name="Control action taken",
        utility=0.80,
        privacy=0.05,
        cognitive_cost=0.10,
    ),
    mst_core.InformationItem(
        item_id="z2",
        name="General operational reason",
        utility=0.70,
        privacy=0.05,
        cognitive_cost=0.15,
    ),
    mst_core.InformationItem(
        item_id="z3",
        name="Current system condition",
        utility=0.50,
        privacy=0.10,
        cognitive_cost=0.20,
    ),
]

module_exact_result = mst_core.exact_mst(
    items=module_test_items,
    tau=0.90,
)

print("Status:", module_exact_result["status"])
print("Selected:", module_exact_result["selected_items"])
print(f"Utility: {module_exact_result['utility']:.4f}")
print(
    f"Disclosure cost: "
    f"{module_exact_result['disclosure_cost']:.4f}"
)

In [ ]:
module_part_4 = r'''

def greedy_mst(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Solve the MST problem using greedy marginal utility
    per marginal disclosure cost.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError(
            "tau must be in the interval (0, 1]."
        )

    total_items = len(items)

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": mst_utility(items),
            "privacy": mst_privacy(items),
            "cognitive_cost": mst_cognitive_cost(items),
            "disclosure_cost": float("inf"),
        }

    selected = []
    remaining = list(items)

    while mst_utility(selected) < tau:

        current_utility = mst_utility(selected)

        current_cost = mst_disclosure_cost(
            selected_items=selected,
            total_items=total_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        best_item = None
        best_score = float("-inf")

        for item in remaining:

            candidate = selected + [item]

            candidate_utility = mst_utility(candidate)

            candidate_cost = mst_disclosure_cost(
                selected_items=candidate,
                total_items=total_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )

            delta_utility = (
                candidate_utility
                - current_utility
            )

            delta_cost = (
                candidate_cost
                - current_cost
            )

            if delta_utility <= 0:
                continue

            score = (
                delta_utility
                / (delta_cost + epsilon)
            )

            if (
                score > best_score
                or (
                    abs(score - best_score) <= 1e-12
                    and best_item is not None
                    and item.item_id < best_item.item_id
                )
            ):
                best_item = item
                best_score = score

        if best_item is None:
            return {
                "status": "INFEASIBLE",
                "selected_items": tuple(
                    item.item_id
                    for item in selected
                ),
                "utility": mst_utility(selected),
                "privacy": mst_privacy(selected),
                "cognitive_cost": mst_cognitive_cost(
                    selected
                ),
                "disclosure_cost": float("inf"),
            }

        selected.append(best_item)
        remaining.remove(best_item)

    final_result = evaluate_subset(
        subset=selected,
        total_items=total_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    return {
        "status": "FEASIBLE",
        "selected_items": final_result["items"],
        "utility": final_result["utility"],
        "privacy": final_result["privacy"],
        "cognitive_cost": final_result[
            "cognitive_cost"
        ],
        "disclosure_cost": final_result[
            "disclosure_cost"
        ],
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_4)

print("Greedy-MST added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available Greedy-MST function:")
print(mst_core.greedy_mst)

In [ ]:
module_greedy_result = mst_core.greedy_mst(
    items=module_test_items,
    tau=0.90,
)

print("Status:", module_greedy_result["status"])
print("Selected:", module_greedy_result["selected_items"])
print(f"Utility: {module_greedy_result['utility']:.4f}")
print(
    f"Disclosure cost: "
    f"{module_greedy_result['disclosure_cost']:.4f}"
)

In [ ]:
module_hard_items = [
    mst_core.InformationItem(
        item_id="h1",
        name="High immediate utility item",
        utility=0.70,
        privacy=0.00,
        cognitive_cost=0.20,
    ),
    mst_core.InformationItem(
        item_id="h2",
        name="Moderate utility item A",
        utility=0.60,
        privacy=0.00,
        cognitive_cost=0.18,
    ),
    mst_core.InformationItem(
        item_id="h3",
        name="Moderate utility item B",
        utility=0.60,
        privacy=0.00,
        cognitive_cost=0.18,
    ),
]

module_exact_hard = mst_core.exact_mst(
    items=module_hard_items,
    tau=0.80,
)

module_greedy_hard = mst_core.greedy_mst(
    items=module_hard_items,
    tau=0.80,
)

print("Exact:", module_exact_hard["selected_items"])
print(
    f"Exact cost: "
    f"{module_exact_hard['disclosure_cost']:.4f}"
)

print("Greedy:", module_greedy_hard["selected_items"])
print(
    f"Greedy cost: "
    f"{module_greedy_hard['disclosure_cost']:.4f}"
)

In [ ]:
module_part_5 = r'''

def progressive_mst_step(
    selected_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Perform one Progressive-MST disclosure step.
    """

    total_items = len(all_items)

    selected_ids = {
        item.item_id
        for item in selected_items
    }

    remaining_items = [
        item
        for item in all_items
        if item.item_id not in selected_ids
    ]

    if not remaining_items:
        return {
            "status": "NO_ADDITIONAL_INFORMATION",
            "selected_items": tuple(
                item.item_id
                for item in selected_items
            ),
        }

    current_utility = mst_utility(
        selected_items
    )

    current_cost = mst_disclosure_cost(
        selected_items=selected_items,
        total_items=total_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    best_item = None
    best_score = float("-inf")

    for item in remaining_items:

        expanded_subset = (
            selected_items + [item]
        )

        new_utility = mst_utility(
            expanded_subset
        )

        new_cost = mst_disclosure_cost(
            selected_items=expanded_subset,
            total_items=total_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        delta_utility = (
            new_utility
            - current_utility
        )

        delta_cost = (
            new_cost
            - current_cost
        )

        if delta_utility <= 0:
            continue

        score = (
            delta_utility
            / (delta_cost + epsilon)
        )

        if (
            score > best_score
            or (
                abs(score - best_score) <= 1e-12
                and best_item is not None
                and item.item_id < best_item.item_id
            )
        ):
            best_item = item
            best_score = score

    if best_item is None:
        return {
            "status": "NO_ADDITIONAL_INFORMATION",
            "selected_items": tuple(
                item.item_id
                for item in selected_items
            ),
        }

    expanded_items = (
        selected_items + [best_item]
    )

    result = evaluate_subset(
        subset=expanded_items,
        total_items=total_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    return {
        "status": "EXPANDED",
        "added_item": best_item.item_id,
        "selected_items": result["items"],
        "utility": result["utility"],
        "privacy": result["privacy"],
        "cognitive_cost": result[
            "cognitive_cost"
        ],
        "disclosure_cost": result[
            "disclosure_cost"
        ],
        "selection_score": best_score,
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_5)

print("Progressive-MST step added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available Progressive-MST function:")
print(mst_core.progressive_mst_step)

In [ ]:
module_progressive_result = mst_core.progressive_mst_step(
    selected_items=[module_test_items[0]],  # Start with z1
    all_items=module_test_items,
)

print("Status:", module_progressive_result["status"])
print("Added item:", module_progressive_result["added_item"])
print("Selected:", module_progressive_result["selected_items"])
print(f"Utility: {module_progressive_result['utility']:.4f}")
print(
    f"Disclosure cost: "
    f"{module_progressive_result['disclosure_cost']:.4f}"
)

In [ ]:
module_part_6 = r'''

def progressive_mst_sequence(
    initial_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> list[dict]:
    """
    Generate the full Progressive-MST disclosure sequence.
    """

    current_items = list(initial_items)

    history = []

    initial_result = evaluate_subset(
        subset=current_items,
        total_items=len(all_items),
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    history.append(
        {
            "step": 0,
            "added_item": None,
            **initial_result,
        }
    )

    step = 1

    while True:

        result = progressive_mst_step(
            selected_items=current_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        if result["status"] != "EXPANDED":
            break

        added_id = result["added_item"]

        added_item = next(
            item
            for item in all_items
            if item.item_id == added_id
        )

        current_items.append(added_item)

        history.append(
            {
                "step": step,
                "added_item": added_id,
                "items": result["selected_items"],
                "utility": result["utility"],
                "privacy": result["privacy"],
                "cognitive_cost": result[
                    "cognitive_cost"
                ],
                "disclosure_cost": result[
                    "disclosure_cost"
                ],
            }
        )

        step += 1

    return history
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_6)

print("Progressive-MST sequence added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available sequence function:")
print(mst_core.progressive_mst_sequence)

In [ ]:
module_progressive_history = mst_core.progressive_mst_sequence(
    initial_items=[module_test_items[0]],
    all_items=module_test_items,
)

for record in module_progressive_history:
    print(
        f"Step {record['step']}: "
        f"Added={record['added_item']}, "
        f"Selected={record['items']}, "
        f"U={record['utility']:.4f}, "
        f"J={record['disclosure_cost']:.4f}"
    )

In [ ]:
module_part_7 = r'''

def compute_optimality_gap(
    exact_result: dict,
    approximate_result: dict,
) -> float:
    """
    Compute the relative optimality gap of an approximate MST solution.

    Gap = (J_approx - J_exact) / J_exact
    """

    if exact_result["status"] != "OPTIMAL":
        raise ValueError(
            "The exact result must contain an optimal solution."
        )

    if approximate_result["status"] not in {
        "FEASIBLE",
        "OPTIMAL",
    }:
        raise ValueError(
            "The approximate result must contain a feasible solution."
        )

    exact_cost = exact_result[
        "disclosure_cost"
    ]

    approximate_cost = approximate_result[
        "disclosure_cost"
    ]

    if exact_cost <= 0:
        if approximate_cost == exact_cost:
            return 0.0

        raise ValueError(
            "Relative optimality gap is undefined "
            "when the exact cost is zero."
        )

    return (
        approximate_cost
        - exact_cost
    ) / exact_cost
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_7)

print("Optimality-gap function added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available optimality-gap function:")
print(mst_core.compute_optimality_gap)

In [ ]:
module_gap = mst_core.compute_optimality_gap(
    exact_result=module_exact_hard,
    approximate_result=module_greedy_hard,
)

print(f"Relative optimality gap: {module_gap:.4f}")
print(f"Optimality gap (%): {100 * module_gap:.2f}%")

In [ ]:
import math
import pandas as pd


validation_results = []


def record_test(test_name, condition, details=""):
    """Record one automated validation check."""
    validation_results.append(
        {
            "test": test_name,
            "passed": bool(condition),
            "details": details,
        }
    )


# --------------------------------------------------
# 1. Core set-function checks
# --------------------------------------------------

record_test(
    "Empty set utility equals zero",
    math.isclose(
        mst_core.mst_utility([]),
        0.0,
        abs_tol=1e-12,
    ),
)

record_test(
    "Empty set privacy equals zero",
    math.isclose(
        mst_core.mst_privacy([]),
        0.0,
        abs_tol=1e-12,
    ),
)

record_test(
    "Empty set cognitive cost equals zero",
    math.isclose(
        mst_core.mst_cognitive_cost([]),
        0.0,
        abs_tol=1e-12,
    ),
)


# --------------------------------------------------
# 2. Exact-MST checks
# --------------------------------------------------

exact_test = mst_core.exact_mst(
    items=module_test_items,
    tau=0.90,
)

record_test(
    "Exact-MST returns optimal status",
    exact_test["status"] == "OPTIMAL",
)

record_test(
    "Exact-MST selects expected subset",
    exact_test["selected_items"] == ("z1", "z2"),
    details=str(exact_test["selected_items"]),
)

record_test(
    "Exact-MST satisfies threshold",
    exact_test["utility"] >= 0.90,
)


# --------------------------------------------------
# 3. Greedy-MST checks
# --------------------------------------------------

greedy_test = mst_core.greedy_mst(
    items=module_test_items,
    tau=0.90,
)

record_test(
    "Greedy-MST returns feasible status",
    greedy_test["status"] == "FEASIBLE",
)

record_test(
    "Greedy-MST satisfies threshold",
    greedy_test["utility"] >= 0.90,
)

record_test(
    "Greedy matches Exact on simple instance",
    greedy_test["selected_items"]
    == exact_test["selected_items"],
)


# --------------------------------------------------
# 4. Infeasibility checks
# --------------------------------------------------

exact_infeasible = mst_core.exact_mst(
    items=module_test_items,
    tau=0.98,
)

greedy_infeasible = mst_core.greedy_mst(
    items=module_test_items,
    tau=0.98,
)

record_test(
    "Exact-MST detects infeasible instance",
    exact_infeasible["status"] == "INFEASIBLE",
)

record_test(
    "Greedy-MST detects infeasible instance",
    greedy_infeasible["status"] == "INFEASIBLE",
)


# --------------------------------------------------
# 5. Exact-vs-Greedy counterexample checks
# --------------------------------------------------

record_test(
    "Exact-MST finds expected hard-instance optimum",
    module_exact_hard["selected_items"]
    == ("h2", "h3"),
)

record_test(
    "Greedy differs from Exact on hard instance",
    module_greedy_hard["selected_items"]
    != module_exact_hard["selected_items"],
)

record_test(
    "Greedy hard-instance cost exceeds optimum",
    module_greedy_hard["disclosure_cost"]
    > module_exact_hard["disclosure_cost"],
)

record_test(
    "Optimality gap is positive",
    module_gap > 0.0,
    details=f"{100 * module_gap:.4f}%",
)


# --------------------------------------------------
# 6. Progressive-MST property checks
# --------------------------------------------------

history = mst_core.progressive_mst_sequence(
    initial_items=[module_test_items[0]],
    all_items=module_test_items,
)

nested = True
utility_monotonic = True
cost_monotonic = True

for previous, current in zip(
    history,
    history[1:],
):
    previous_set = set(previous["items"])
    current_set = set(current["items"])

    if not previous_set.issubset(current_set):
        nested = False

    if current["utility"] < previous["utility"] - 1e-12:
        utility_monotonic = False

    if (
        current["disclosure_cost"]
        < previous["disclosure_cost"] - 1e-12
    ):
        cost_monotonic = False


record_test(
    "Progressive disclosure is nested",
    nested,
)

record_test(
    "Progressive utility is non-decreasing",
    utility_monotonic,
)

record_test(
    "Progressive disclosure cost is non-decreasing",
    cost_monotonic,
)


# --------------------------------------------------
# 7. Input validation checks
# --------------------------------------------------

invalid_item_rejected = False

try:
    mst_core.InformationItem(
        item_id="invalid",
        name="Invalid item",
        utility=1.50,
        privacy=0.10,
        cognitive_cost=0.10,
    )
except ValueError:
    invalid_item_rejected = True

record_test(
    "Invalid item utility is rejected",
    invalid_item_rejected,
)


invalid_tau_rejected = False

try:
    mst_core.exact_mst(
        items=module_test_items,
        tau=1.50,
    )
except ValueError:
    invalid_tau_rejected = True

record_test(
    "Invalid sufficiency threshold is rejected",
    invalid_tau_rejected,
)


# --------------------------------------------------
# Create and save validation report
# --------------------------------------------------

validation_report = pd.DataFrame(
    validation_results
)

validation_output = (
    RESULTS_DIR
    / "mst_core_automated_validation.csv"
)

validation_report.to_csv(
    validation_output,
    index=False,
)

print(validation_report.to_string(index=False))

total_tests = len(validation_report)
passed_tests = int(
    validation_report["passed"].sum()
)

print(
    f"\nPassed: {passed_tests}/{total_tests}"
)

print(
    "Overall status:",
    "ALL TESTS PASSED"
    if passed_tests == total_tests
    else "SOME TESTS FAILED",
)

print("\nSaved to:")
print(validation_output)

In [ ]:
module_part_8 = r'''

def mst_normalized_cognitive_cost(
    selected_items: Iterable[InformationItem],
    all_items: Iterable[InformationItem],
) -> float:
    """
    Normalize the interpretation cost of subset S relative to
    the total interpretation cost of the full admissible set D.

    C_bar(S) = C(S) / C(D), if C(D) > 0
             = 0, otherwise.
    """

    selected = list(selected_items)
    admissible = list(all_items)

    selected_cost = mst_cognitive_cost(selected)
    total_cost = mst_cognitive_cost(admissible)

    if total_cost == 0.0:
        return 0.0

    return selected_cost / total_cost
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_8)

print("Normalized interpretation-cost function added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available normalization function:")
print(mst_core.mst_normalized_cognitive_cost)

In [ ]:
normalized_cost_all = (
    mst_core.mst_normalized_cognitive_cost(
        selected_items=module_test_items,
        all_items=module_test_items,
    )
)

normalized_cost_z1 = (
    mst_core.mst_normalized_cognitive_cost(
        selected_items=[module_test_items[0]],
        all_items=module_test_items,
    )
)

normalized_cost_z1_z2 = (
    mst_core.mst_normalized_cognitive_cost(
        selected_items=module_test_items[:2],
        all_items=module_test_items,
    )
)

print(
    f"Normalized cost of full set: "
    f"{normalized_cost_all:.4f}"
)

print(
    f"Normalized cost of z1: "
    f"{normalized_cost_z1:.4f}"
)

print(
    f"Normalized cost of z1+z2: "
    f"{normalized_cost_z1_z2:.4f}"
)

In [ ]:
module_part_9 = r'''

def mst_normalized_disclosure_cost(
    selected_items: Iterable[InformationItem],
    all_items: Iterable[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> float:
    """
    Revised normalized MST disclosure objective.

    J(S) =
        lambda_privacy * P(S)
        + lambda_cognitive * C_bar(S)
        + lambda_cardinality * |S| / |D|

    where:
        C_bar(S) = C(S) / C(D)
    """

    selected = list(selected_items)
    admissible = list(all_items)

    total_items = len(admissible)

    if total_items <= 0:
        raise ValueError(
            "all_items must contain at least one admissible item."
        )

    if any(
        weight < 0
        for weight in (
            lambda_privacy,
            lambda_cognitive,
            lambda_cardinality,
        )
    ):
        raise ValueError(
            "All lambda weights must be non-negative."
        )

    privacy_component = mst_privacy(selected)

    normalized_cognitive_component = (
        mst_normalized_cognitive_cost(
            selected_items=selected,
            all_items=admissible,
        )
    )

    cardinality_component = (
        len(selected) / total_items
    )

    return (
        lambda_privacy * privacy_component
        + lambda_cognitive * normalized_cognitive_component
        + lambda_cardinality * cardinality_component
    )
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_9)

print("Normalized MST objective added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available normalized objective:")
print(mst_core.mst_normalized_disclosure_cost)

In [ ]:
normalized_j_z1 = (
    mst_core.mst_normalized_disclosure_cost(
        selected_items=[module_test_items[0]],
        all_items=module_test_items,
    )
)

normalized_j_z1_z2 = (
    mst_core.mst_normalized_disclosure_cost(
        selected_items=module_test_items[:2],
        all_items=module_test_items,
    )
)

normalized_j_all = (
    mst_core.mst_normalized_disclosure_cost(
        selected_items=module_test_items,
        all_items=module_test_items,
    )
)

print(f"J_normalized(z1) = {normalized_j_z1:.4f}")
print(
    f"J_normalized(z1,z2) = "
    f"{normalized_j_z1_z2:.4f}"
)
print(
    f"J_normalized(full set) = "
    f"{normalized_j_all:.4f}"
)

In [ ]:
module_part_10 = r'''

def evaluate_subset_normalized(
    subset: Iterable[InformationItem],
    all_items: Iterable[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Evaluate one disclosure subset using the normalized MST objective.
    """

    selected = list(subset)
    admissible = list(all_items)

    return {
        "items": tuple(
            item.item_id
            for item in selected
        ),
        "utility": mst_utility(selected),
        "privacy": mst_privacy(selected),
        "cognitive_cost": mst_cognitive_cost(selected),
        "normalized_cognitive_cost":
            mst_normalized_cognitive_cost(
                selected_items=selected,
                all_items=admissible,
            ),
        "disclosure_cost":
            mst_normalized_disclosure_cost(
                selected_items=selected,
                all_items=admissible,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            ),
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_10)

print("Normalized subset evaluator added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available normalized evaluator:")
print(mst_core.evaluate_subset_normalized)

In [ ]:
normalized_subset_test = (
    mst_core.evaluate_subset_normalized(
        subset=module_test_items[:2],
        all_items=module_test_items,
    )
)

for key, value in normalized_subset_test.items():
    print(f"{key}: {value}")

In [ ]:
module_part_11 = r'''

def exact_mst_normalized(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Solve the MST problem exactly using the normalized disclosure objective.

    Returns the globally minimum-cost subset satisfying U(S) >= tau.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError(
            "tau must be in the interval (0, 1]."
        )

    total_items = len(items)

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "normalized_cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    # Under monotone utility, the full admissible set provides
    # the maximum attainable utility.
    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": mst_utility(items),
            "privacy": mst_privacy(items),
            "cognitive_cost": mst_cognitive_cost(items),
            "normalized_cognitive_cost":
                mst_normalized_cognitive_cost(
                    selected_items=items,
                    all_items=items,
                ),
            "disclosure_cost": float("inf"),
        }

    best_result = None

    for subset_size in range(total_items + 1):

        for subset in combinations(
            items,
            subset_size,
        ):

            result = evaluate_subset_normalized(
                subset=subset,
                all_items=items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )

            if result["utility"] < tau:
                continue

            if best_result is None:
                best_result = result
                continue

            current_key = (
                result["disclosure_cost"],
                result["privacy"],
                result["normalized_cognitive_cost"],
                len(result["items"]),
                result["items"],
            )

            best_key = (
                best_result["disclosure_cost"],
                best_result["privacy"],
                best_result["normalized_cognitive_cost"],
                len(best_result["items"]),
                best_result["items"],
            )

            if current_key < best_key:
                best_result = result

    return {
        "status": "OPTIMAL",
        "selected_items": best_result["items"],
        "utility": best_result["utility"],
        "privacy": best_result["privacy"],
        "cognitive_cost": best_result["cognitive_cost"],
        "normalized_cognitive_cost":
            best_result["normalized_cognitive_cost"],
        "disclosure_cost":
            best_result["disclosure_cost"],
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_11)

print("Normalized Exact-MST added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available normalized Exact-MST:")
print(mst_core.exact_mst_normalized)

In [ ]:
normalized_exact_result = (
    mst_core.exact_mst_normalized(
        items=module_test_items,
        tau=0.90,
    )
)

print("Status:", normalized_exact_result["status"])
print("Selected:", normalized_exact_result["selected_items"])
print(f"Utility: {normalized_exact_result['utility']:.4f}")
print(
    "Normalized cognitive cost:",
    f"{normalized_exact_result['normalized_cognitive_cost']:.4f}",
)
print(
    "Disclosure cost:",
    f"{normalized_exact_result['disclosure_cost']:.4f}",
)

In [ ]:
module_part_12 = r'''

def greedy_mst_normalized(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Solve MST greedily using marginal utility per marginal
    normalized disclosure cost.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError(
            "tau must be in the interval (0, 1]."
        )

    total_items = len(items)

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "normalized_cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": mst_utility(items),
            "privacy": mst_privacy(items),
            "cognitive_cost": mst_cognitive_cost(items),
            "normalized_cognitive_cost":
                mst_normalized_cognitive_cost(
                    selected_items=items,
                    all_items=items,
                ),
            "disclosure_cost": float("inf"),
        }

    selected = []
    remaining = list(items)

    while mst_utility(selected) < tau:

        current_utility = mst_utility(selected)

        current_cost = mst_normalized_disclosure_cost(
            selected_items=selected,
            all_items=items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        best_item = None
        best_score = float("-inf")

        for item in remaining:

            candidate = selected + [item]

            candidate_utility = mst_utility(
                candidate
            )

            candidate_cost = (
                mst_normalized_disclosure_cost(
                    selected_items=candidate,
                    all_items=items,
                    lambda_privacy=lambda_privacy,
                    lambda_cognitive=lambda_cognitive,
                    lambda_cardinality=lambda_cardinality,
                )
            )

            delta_utility = (
                candidate_utility
                - current_utility
            )

            delta_cost = (
                candidate_cost
                - current_cost
            )

            if delta_utility <= 0:
                continue

            score = (
                delta_utility
                / (delta_cost + epsilon)
            )

            if (
                score > best_score
                or (
                    abs(score - best_score) <= 1e-12
                    and best_item is not None
                    and item.item_id < best_item.item_id
                )
            ):
                best_item = item
                best_score = score

        if best_item is None:
            return {
                "status": "INFEASIBLE",
                "selected_items": tuple(
                    item.item_id
                    for item in selected
                ),
                "utility": mst_utility(selected),
                "privacy": mst_privacy(selected),
                "cognitive_cost":
                    mst_cognitive_cost(selected),
                "normalized_cognitive_cost":
                    mst_normalized_cognitive_cost(
                        selected_items=selected,
                        all_items=items,
                    ),
                "disclosure_cost": float("inf"),
            }

        selected.append(best_item)
        remaining.remove(best_item)

    final_result = evaluate_subset_normalized(
        subset=selected,
        all_items=items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    return {
        "status": "FEASIBLE",
        "selected_items": final_result["items"],
        "utility": final_result["utility"],
        "privacy": final_result["privacy"],
        "cognitive_cost":
            final_result["cognitive_cost"],
        "normalized_cognitive_cost":
            final_result["normalized_cognitive_cost"],
        "disclosure_cost":
            final_result["disclosure_cost"],
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_12)

print("Normalized Greedy-MST added to:")
print(module_path)

In [ ]:
importlib.reload(mst_core)

print("Available normalized Greedy-MST:")
print(mst_core.greedy_mst_normalized)

In [ ]:
normalized_greedy_result = (
    mst_core.greedy_mst_normalized(
        items=module_test_items,
        tau=0.90,
    )
)

print("Status:", normalized_greedy_result["status"])
print("Selected:", normalized_greedy_result["selected_items"])
print(f"Utility: {normalized_greedy_result['utility']:.4f}")
print(
    "Normalized cognitive cost:",
    f"{normalized_greedy_result['normalized_cognitive_cost']:.4f}",
)
print(
    "Disclosure cost:",
    f"{normalized_greedy_result['disclosure_cost']:.4f}",
)

In [ ]:
normalized_exact_hard = (
    mst_core.exact_mst_normalized(
        items=module_hard_items,
        tau=0.80,
    )
)

normalized_greedy_hard = (
    mst_core.greedy_mst_normalized(
        items=module_hard_items,
        tau=0.80,
    )
)

print("EXACT-MST")
print("Selected:", normalized_exact_hard["selected_items"])
print(
    "Disclosure cost:",
    f"{normalized_exact_hard['disclosure_cost']:.4f}",
)

print("\nGREEDY-MST")
print("Selected:", normalized_greedy_hard["selected_items"])
print(
    "Disclosure cost:",
    f"{normalized_greedy_hard['disclosure_cost']:.4f}",
)

In [ ]:
normalized_hard_gap = (
    mst_core.compute_optimality_gap(
        exact_result=normalized_exact_hard,
        approximate_result=normalized_greedy_hard,
    )
)

print(
    f"Relative optimality gap: "
    f"{normalized_hard_gap:.4f}"
)

print(
    f"Optimality gap (%): "
    f"{100 * normalized_hard_gap:.2f}%"
)

In [ ]:
module_part_13 = r'''

def progressive_mst_step_normalized(
    selected_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Perform one Progressive-MST disclosure step using
    the normalized MST disclosure objective.
    """

    selected_ids = {
        item.item_id
        for item in selected_items
    }

    remaining_items = [
        item
        for item in all_items
        if item.item_id not in selected_ids
    ]

    if not remaining_items:
        return {
            "status": "NO_ADDITIONAL_INFORMATION",
            "selected_items": tuple(
                item.item_id
                for item in selected_items
            ),
        }

    current_utility = mst_utility(
        selected_items
    )

    current_cost = (
        mst_normalized_disclosure_cost(
            selected_items=selected_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )
    )

    best_item = None
    best_score = float("-inf")

    for item in remaining_items:

        expanded_subset = (
            selected_items + [item]
        )

        new_utility = mst_utility(
            expanded_subset
        )

        new_cost = (
            mst_normalized_disclosure_cost(
                selected_items=expanded_subset,
                all_items=all_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )
        )

        delta_utility = (
            new_utility
            - current_utility
        )

        delta_cost = (
            new_cost
            - current_cost
        )

        if delta_utility <= 0:
            continue

        score = (
            delta_utility
            / (delta_cost + epsilon)
        )

        if (
            score > best_score
            or (
                abs(score - best_score) <= 1e-12
                and best_item is not None
                and item.item_id < best_item.item_id
            )
        ):
            best_item = item
            best_score = score

    if best_item is None:
        return {
            "status": "NO_ADDITIONAL_INFORMATION",
            "selected_items": tuple(
                item.item_id
                for item in selected_items
            ),
        }

    expanded_items = (
        selected_items + [best_item]
    )

    result = evaluate_subset_normalized(
        subset=expanded_items,
        all_items=all_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    return {
        "status": "EXPANDED",
        "added_item": best_item.item_id,
        "selected_items": result["items"],
        "utility": result["utility"],
        "privacy": result["privacy"],
        "cognitive_cost":
            result["cognitive_cost"],
        "normalized_cognitive_cost":
            result["normalized_cognitive_cost"],
        "disclosure_cost":
            result["disclosure_cost"],
        "selection_score": best_score,
    }
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_13)

print(
    "Normalized Progressive-MST step added to:"
)
print(module_path)

In [ ]:
importlib.reload(mst_core)

print(
    "Available normalized Progressive-MST:"
)
print(
    mst_core.progressive_mst_step_normalized
)

In [ ]:
normalized_progressive_result = (
    mst_core.progressive_mst_step_normalized(
        selected_items=[
            module_test_items[0]
        ],
        all_items=module_test_items,
    )
)

print(
    "Status:",
    normalized_progressive_result["status"]
)

print(
    "Added item:",
    normalized_progressive_result["added_item"]
)

print(
    "Selected:",
    normalized_progressive_result["selected_items"]
)

print(
    "Utility:",
    f"{normalized_progressive_result['utility']:.4f}"
)

print(
    "Normalized cognitive cost:",
    f"{normalized_progressive_result['normalized_cognitive_cost']:.4f}"
)

print(
    "Disclosure cost:",
    f"{normalized_progressive_result['disclosure_cost']:.4f}"
)

In [ ]:
module_part_14 = r'''

def progressive_mst_sequence_normalized(
    initial_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> list[dict]:
    """
    Generate the full Progressive-MST disclosure sequence
    using the normalized disclosure objective.
    """

    current_items = list(initial_items)
    history = []

    initial_result = evaluate_subset_normalized(
        subset=current_items,
        all_items=all_items,
        lambda_privacy=lambda_privacy,
        lambda_cognitive=lambda_cognitive,
        lambda_cardinality=lambda_cardinality,
    )

    history.append(
        {
            "step": 0,
            "added_item": None,
            **initial_result,
        }
    )

    step = 1

    while True:

        result = progressive_mst_step_normalized(
            selected_items=current_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )

        if result["status"] != "EXPANDED":
            break

        added_id = result["added_item"]

        added_item = next(
            item
            for item in all_items
            if item.item_id == added_id
        )

        current_items.append(added_item)

        history.append(
            {
                "step": step,
                "added_item": added_id,
                "items": result["selected_items"],
                "utility": result["utility"],
                "privacy": result["privacy"],
                "cognitive_cost":
                    result["cognitive_cost"],
                "normalized_cognitive_cost":
                    result["normalized_cognitive_cost"],
                "disclosure_cost":
                    result["disclosure_cost"],
            }
        )

        step += 1

    return history
'''

with module_path.open(
    "a",
    encoding="utf-8",
) as file:
    file.write(module_part_14)

print(
    "Normalized Progressive-MST sequence added to:"
)
print(module_path)

In [ ]:
importlib.reload(mst_core)

print(
    "Available normalized sequence function:"
)
print(
    mst_core.progressive_mst_sequence_normalized
)

In [ ]:
normalized_progressive_history = (
    mst_core.progressive_mst_sequence_normalized(
        initial_items=[module_test_items[0]],
        all_items=module_test_items,
    )
)

for record in normalized_progressive_history:
    print(
        f"Step {record['step']}: "
        f"Added={record['added_item']}, "
        f"Selected={record['items']}, "
        f"U={record['utility']:.4f}, "
        f"C_bar={record['normalized_cognitive_cost']:.4f}, "
        f"J={record['disclosure_cost']:.4f}"
    )

In [ ]:
normalized_property_results = []

nested_disclosure = True
monotonic_utility = True
monotonic_cost = True

for previous, current in zip(
    normalized_progressive_history,
    normalized_progressive_history[1:],
):

    previous_items = set(
        previous["items"]
    )

    current_items = set(
        current["items"]
    )

    # Nested disclosure:
    # S^(k) subset of S^(k+1)
    if not previous_items.issubset(
        current_items
    ):
        nested_disclosure = False

    # Monotonic utility:
    # U(S^(k)) <= U(S^(k+1))
    if (
        current["utility"]
        < previous["utility"] - 1e-12
    ):
        monotonic_utility = False

    # Monotonic normalized disclosure cost:
    # J(S^(k)) <= J(S^(k+1))
    if (
        current["disclosure_cost"]
        < previous["disclosure_cost"] - 1e-12
    ):
        monotonic_cost = False


normalized_property_results = {
    "nested_disclosure":
        nested_disclosure,

    "monotonic_utility":
        monotonic_utility,

    "monotonic_normalized_disclosure_cost":
        monotonic_cost,

    "all_properties_satisfied":
        (
            nested_disclosure
            and monotonic_utility
            and monotonic_cost
        ),
}


for property_name, passed in (
    normalized_property_results.items()
):
    print(
        f"{property_name}: {passed}"
    )

In [ ]:
normalized_validation_df = pd.DataFrame(
    [
        {
            "property": key,
            "passed": value,
        }
        for key, value in normalized_property_results.items()
    ]
)

normalized_validation_output = (
    RESULTS_DIR
    / "normalized_mst_property_validation.csv"
)

normalized_validation_df.to_csv(
    normalized_validation_output,
    index=False,
)


normalized_sequence_df = pd.DataFrame(
    normalized_progressive_history
)

normalized_sequence_output = (
    RESULTS_DIR
    / "normalized_progressive_mst_sequence.csv"
)

normalized_sequence_df.to_csv(
    normalized_sequence_output,
    index=False,
)


print("Saved successfully:")
print(normalized_validation_output)
print(normalized_sequence_output)

In [ ]:
from shutil import copy2

current_module = (
    PYTHON_DIR / "mst_core.py"
)

backup_module = (
    PYTHON_DIR / "mst_core_development_backup.py"
)

copy2(
    current_module,
    backup_module,
)

print("Development module backed up successfully:")
print(backup_module)

In [ ]:
clean_module = r'''
from __future__ import annotations

from collections.abc import Iterable
from dataclasses import dataclass
from itertools import combinations


@dataclass(frozen=True)
class InformationItem:
    """
    A privacy-admissible information item considered by MST.

    Parameters
    ----------
    item_id : str
        Unique identifier of the information item.
    name : str
        Human-readable description.
    utility : float
        Modeled decision-support contribution in [0, 1].
    privacy : float
        Modeled privacy exposure in [0, 1].
    cognitive_cost : float
        Modeled interpretation cost, constrained to be non-negative.
    """

    item_id: str
    name: str
    utility: float
    privacy: float
    cognitive_cost: float

    def __post_init__(self) -> None:

        if not self.item_id:
            raise ValueError(
                "item_id must be a non-empty string."
            )

        if not 0.0 <= self.utility <= 1.0:
            raise ValueError(
                f"utility must be in [0, 1], "
                f"received {self.utility}"
            )

        if not 0.0 <= self.privacy <= 1.0:
            raise ValueError(
                f"privacy must be in [0, 1], "
                f"received {self.privacy}"
            )

        if self.cognitive_cost < 0.0:
            raise ValueError(
                "cognitive_cost must be non-negative."
            )


def _validate_unique_ids(
    items: Iterable[InformationItem],
) -> None:
    """
    Ensure that information identifiers are unique.
    """

    item_list = list(items)

    ids = [
        item.item_id
        for item in item_list
    ]

    if len(ids) != len(set(ids)):
        raise ValueError(
            "All information items must have unique item_id values."
        )


def mst_utility(
    items: Iterable[InformationItem],
) -> float:
    """
    Bounded cumulative modeled decision-support utility.

    U(S) = 1 - product(1 - u(z))
    """

    product = 1.0

    for item in items:
        product *= (
            1.0 - item.utility
        )

    return 1.0 - product


def mst_privacy(
    items: Iterable[InformationItem],
) -> float:
    """
    Bounded cumulative privacy exposure.

    P(S) = 1 - product(1 - p(z))
    """

    product = 1.0

    for item in items:
        product *= (
            1.0 - item.privacy
        )

    return 1.0 - product


def mst_cognitive_cost(
    items: Iterable[InformationItem],
) -> float:
    """
    Additive modeled interpretation cost.

    C(S) = sum c(z)
    """

    return sum(
        item.cognitive_cost
        for item in items
    )


def mst_normalized_cognitive_cost(
    selected_items: Iterable[InformationItem],
    all_items: Iterable[InformationItem],
) -> float:
    """
    Normalized modeled interpretation cost.

    C_bar(S) = C(S) / C(D), if C(D) > 0,
               0 otherwise.
    """

    selected = list(selected_items)
    admissible = list(all_items)

    selected_cost = (
        mst_cognitive_cost(selected)
    )

    total_cost = (
        mst_cognitive_cost(admissible)
    )

    if total_cost == 0.0:
        return 0.0

    return (
        selected_cost
        / total_cost
    )


def mst_disclosure_cost(
    selected_items: Iterable[InformationItem],
    all_items: Iterable[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> float:
    """
    Official normalized MST disclosure objective.

    J(S) =
        lambda_privacy * P(S)
        + lambda_cognitive * C_bar(S)
        + lambda_cardinality * |S| / |D|
    """

    selected = list(
        selected_items
    )

    admissible = list(
        all_items
    )

    total_items = len(
        admissible
    )

    if total_items == 0:
        raise ValueError(
            "all_items must contain at least one "
            "admissible information item."
        )

    if any(
        weight < 0.0
        for weight in (
            lambda_privacy,
            lambda_cognitive,
            lambda_cardinality,
        )
    ):
        raise ValueError(
            "All lambda weights must be non-negative."
        )

    privacy_component = (
        mst_privacy(selected)
    )

    cognitive_component = (
        mst_normalized_cognitive_cost(
            selected_items=selected,
            all_items=admissible,
        )
    )

    cardinality_component = (
        len(selected)
        / total_items
    )

    return (
        lambda_privacy
        * privacy_component
        +
        lambda_cognitive
        * cognitive_component
        +
        lambda_cardinality
        * cardinality_component
    )


def evaluate_subset(
    subset: Iterable[InformationItem],
    all_items: Iterable[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Evaluate one disclosure subset under the official
    normalized MST formulation.
    """

    selected = list(
        subset
    )

    admissible = list(
        all_items
    )

    return {
        "items": tuple(
            item.item_id
            for item in selected
        ),
        "utility":
            mst_utility(selected),
        "privacy":
            mst_privacy(selected),
        "cognitive_cost":
            mst_cognitive_cost(selected),
        "normalized_cognitive_cost":
            mst_normalized_cognitive_cost(
                selected_items=selected,
                all_items=admissible,
            ),
        "disclosure_cost":
            mst_disclosure_cost(
                selected_items=selected,
                all_items=admissible,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            ),
    }


def exact_mst(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> dict:
    """
    Solve MST exactly by exhaustive subset enumeration.

    Returns the globally minimum-cost disclosure subset
    satisfying U(S) >= tau.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError(
            "tau must be in the interval (0, 1]."
        )

    _validate_unique_ids(
        items
    )

    total_items = len(
        items
    )

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "normalized_cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility":
                mst_utility(items),
            "privacy":
                mst_privacy(items),
            "cognitive_cost":
                mst_cognitive_cost(items),
            "normalized_cognitive_cost":
                mst_normalized_cognitive_cost(
                    selected_items=items,
                    all_items=items,
                ),
            "disclosure_cost":
                float("inf"),
        }

    best_result = None

    for subset_size in range(
        total_items + 1
    ):

        for subset in combinations(
            items,
            subset_size,
        ):

            result = evaluate_subset(
                subset=subset,
                all_items=items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )

            if (
                result["utility"]
                < tau
            ):
                continue

            if best_result is None:
                best_result = result
                continue

            current_key = (
                result["disclosure_cost"],
                result["privacy"],
                result[
                    "normalized_cognitive_cost"
                ],
                len(result["items"]),
                result["items"],
            )

            best_key = (
                best_result[
                    "disclosure_cost"
                ],
                best_result[
                    "privacy"
                ],
                best_result[
                    "normalized_cognitive_cost"
                ],
                len(
                    best_result["items"]
                ),
                best_result[
                    "items"
                ],
            )

            if current_key < best_key:
                best_result = result

    return {
        "status": "OPTIMAL",
        "selected_items":
            best_result["items"],
        "utility":
            best_result["utility"],
        "privacy":
            best_result["privacy"],
        "cognitive_cost":
            best_result[
                "cognitive_cost"
            ],
        "normalized_cognitive_cost":
            best_result[
                "normalized_cognitive_cost"
            ],
        "disclosure_cost":
            best_result[
                "disclosure_cost"
            ],
    }


def greedy_mst(
    items: list[InformationItem],
    tau: float,
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Solve MST greedily using marginal modeled utility
    per marginal normalized disclosure cost.
    """

    if not 0.0 < tau <= 1.0:
        raise ValueError(
            "tau must be in the interval (0, 1]."
        )

    _validate_unique_ids(
        items
    )

    total_items = len(
        items
    )

    if total_items == 0:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility": 0.0,
            "privacy": 0.0,
            "cognitive_cost": 0.0,
            "normalized_cognitive_cost": 0.0,
            "disclosure_cost": float("inf"),
        }

    if mst_utility(items) < tau:
        return {
            "status": "INFEASIBLE",
            "selected_items": tuple(),
            "utility":
                mst_utility(items),
            "privacy":
                mst_privacy(items),
            "cognitive_cost":
                mst_cognitive_cost(items),
            "normalized_cognitive_cost":
                mst_normalized_cognitive_cost(
                    selected_items=items,
                    all_items=items,
                ),
            "disclosure_cost":
                float("inf"),
        }

    selected = []
    remaining = list(
        items
    )

    while (
        mst_utility(selected)
        < tau
    ):

        current_utility = (
            mst_utility(selected)
        )

        current_cost = (
            mst_disclosure_cost(
                selected_items=selected,
                all_items=items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )
        )

        best_item = None
        best_score = float(
            "-inf"
        )

        for item in remaining:

            candidate = (
                selected + [item]
            )

            candidate_utility = (
                mst_utility(
                    candidate
                )
            )

            candidate_cost = (
                mst_disclosure_cost(
                    selected_items=candidate,
                    all_items=items,
                    lambda_privacy=lambda_privacy,
                    lambda_cognitive=lambda_cognitive,
                    lambda_cardinality=lambda_cardinality,
                )
            )

            delta_utility = (
                candidate_utility
                - current_utility
            )

            delta_cost = (
                candidate_cost
                - current_cost
            )

            if delta_utility <= 0.0:
                continue

            score = (
                delta_utility
                / (
                    delta_cost
                    + epsilon
                )
            )

            if (
                score > best_score
                or (
                    abs(
                        score
                        - best_score
                    )
                    <= 1e-12
                    and best_item
                    is not None
                    and item.item_id
                    < best_item.item_id
                )
            ):
                best_item = item
                best_score = score

        if best_item is None:
            return {
                "status":
                    "INFEASIBLE",
                "selected_items":
                    tuple(
                        item.item_id
                        for item
                        in selected
                    ),
                "utility":
                    mst_utility(
                        selected
                    ),
                "privacy":
                    mst_privacy(
                        selected
                    ),
                "cognitive_cost":
                    mst_cognitive_cost(
                        selected
                    ),
                "normalized_cognitive_cost":
                    mst_normalized_cognitive_cost(
                        selected_items=selected,
                        all_items=items,
                    ),
                "disclosure_cost":
                    float("inf"),
            }

        selected.append(
            best_item
        )

        remaining.remove(
            best_item
        )

    final_result = (
        evaluate_subset(
            subset=selected,
            all_items=items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )
    )

    return {
        "status": "FEASIBLE",
        "selected_items":
            final_result["items"],
        "utility":
            final_result["utility"],
        "privacy":
            final_result["privacy"],
        "cognitive_cost":
            final_result[
                "cognitive_cost"
            ],
        "normalized_cognitive_cost":
            final_result[
                "normalized_cognitive_cost"
            ],
        "disclosure_cost":
            final_result[
                "disclosure_cost"
            ],
    }


def progressive_mst_step(
    selected_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
    epsilon: float = 1e-12,
) -> dict:
    """
    Perform one Progressive-MST disclosure step.
    """

    _validate_unique_ids(
        all_items
    )

    selected_ids = {
        item.item_id
        for item
        in selected_items
    }

    remaining_items = [
        item
        for item in all_items
        if item.item_id
        not in selected_ids
    ]

    if not remaining_items:
        return {
            "status":
                "NO_ADDITIONAL_INFORMATION",
            "selected_items":
                tuple(
                    item.item_id
                    for item
                    in selected_items
                ),
        }

    current_utility = (
        mst_utility(
            selected_items
        )
    )

    current_cost = (
        mst_disclosure_cost(
            selected_items=selected_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )
    )

    best_item = None
    best_score = float(
        "-inf"
    )

    for item in remaining_items:

        expanded_subset = (
            selected_items
            + [item]
        )

        new_utility = (
            mst_utility(
                expanded_subset
            )
        )

        new_cost = (
            mst_disclosure_cost(
                selected_items=expanded_subset,
                all_items=all_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )
        )

        delta_utility = (
            new_utility
            - current_utility
        )

        delta_cost = (
            new_cost
            - current_cost
        )

        if delta_utility <= 0.0:
            continue

        score = (
            delta_utility
            / (
                delta_cost
                + epsilon
            )
        )

        if (
            score > best_score
            or (
                abs(
                    score
                    - best_score
                )
                <= 1e-12
                and best_item
                is not None
                and item.item_id
                < best_item.item_id
            )
        ):
            best_item = item
            best_score = score

    if best_item is None:
        return {
            "status":
                "NO_ADDITIONAL_INFORMATION",
            "selected_items":
                tuple(
                    item.item_id
                    for item
                    in selected_items
                ),
        }

    expanded_items = (
        selected_items
        + [best_item]
    )

    result = (
        evaluate_subset(
            subset=expanded_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )
    )

    return {
        "status": "EXPANDED",
        "added_item":
            best_item.item_id,
        "selected_items":
            result["items"],
        "utility":
            result["utility"],
        "privacy":
            result["privacy"],
        "cognitive_cost":
            result[
                "cognitive_cost"
            ],
        "normalized_cognitive_cost":
            result[
                "normalized_cognitive_cost"
            ],
        "disclosure_cost":
            result[
                "disclosure_cost"
            ],
        "selection_score":
            best_score,
    }


def progressive_mst_sequence(
    initial_items: list[InformationItem],
    all_items: list[InformationItem],
    lambda_privacy: float = 1.0,
    lambda_cognitive: float = 1.0,
    lambda_cardinality: float = 0.01,
) -> list[dict]:
    """
    Generate the complete Progressive-MST disclosure sequence.
    """

    _validate_unique_ids(
        all_items
    )

    current_items = list(
        initial_items
    )

    history = []

    initial_result = (
        evaluate_subset(
            subset=current_items,
            all_items=all_items,
            lambda_privacy=lambda_privacy,
            lambda_cognitive=lambda_cognitive,
            lambda_cardinality=lambda_cardinality,
        )
    )

    history.append(
        {
            "step": 0,
            "added_item": None,
            **initial_result,
        }
    )

    step = 1

    while True:

        result = (
            progressive_mst_step(
                selected_items=current_items,
                all_items=all_items,
                lambda_privacy=lambda_privacy,
                lambda_cognitive=lambda_cognitive,
                lambda_cardinality=lambda_cardinality,
            )
        )

        if (
            result["status"]
            != "EXPANDED"
        ):
            break

        added_id = (
            result["added_item"]
        )

        added_item = next(
            item
            for item in all_items
            if item.item_id
            == added_id
        )

        current_items.append(
            added_item
        )

        history.append(
            {
                "step":
                    step,
                "added_item":
                    added_id,
                "items":
                    result[
                        "selected_items"
                    ],
                "utility":
                    result[
                        "utility"
                    ],
                "privacy":
                    result[
                        "privacy"
                    ],
                "cognitive_cost":
                    result[
                        "cognitive_cost"
                    ],
                "normalized_cognitive_cost":
                    result[
                        "normalized_cognitive_cost"
                    ],
                "disclosure_cost":
                    result[
                        "disclosure_cost"
                    ],
            }
        )

        step += 1

    return history


def compute_optimality_gap(
    exact_result: dict,
    approximate_result: dict,
) -> float:
    """
    Compute the relative optimality gap.

    Gap =
        (J_approx - J_exact)
        / J_exact
    """

    if (
        exact_result["status"]
        != "OPTIMAL"
    ):
        raise ValueError(
            "The exact result must contain "
            "an optimal solution."
        )

    if (
        approximate_result[
            "status"
        ]
        not in {
            "FEASIBLE",
            "OPTIMAL",
        }
    ):
        raise ValueError(
            "The approximate result must "
            "contain a feasible solution."
        )

    exact_cost = (
        exact_result[
            "disclosure_cost"
        ]
    )

    approximate_cost = (
        approximate_result[
            "disclosure_cost"
        ]
    )

    if exact_cost <= 0.0:

        if (
            approximate_cost
            == exact_cost
        ):
            return 0.0

        raise ValueError(
            "Relative optimality gap is "
            "undefined when the exact "
            "optimal cost is zero."
        )

    return (
        approximate_cost
        - exact_cost
    ) / exact_cost
'''

clean_module_path = (
    PYTHON_DIR / "mst_core.py"
)

clean_module_path.write_text(
    clean_module.strip() + "\n",
    encoding="utf-8",
)

print("Clean official MST module created:")
print(clean_module_path)

In [ ]:
import importlib
import mst_core

mst_core = importlib.reload(mst_core)

print("Loaded module:")
print(mst_core.__file__)

print("\nCore functions available:")
print("InformationItem:", hasattr(mst_core, "InformationItem"))
print("Exact-MST:", hasattr(mst_core, "exact_mst"))
print("Greedy-MST:", hasattr(mst_core, "greedy_mst"))
print("Progressive-MST:", hasattr(mst_core, "progressive_mst_sequence"))
print("Optimality gap:", hasattr(mst_core, "compute_optimality_gap"))

In [ ]:
clean_test_items = [
    mst_core.InformationItem(
        item_id="z1",
        name="Control action taken",
        utility=0.80,
        privacy=0.05,
        cognitive_cost=0.10,
    ),
    mst_core.InformationItem(
        item_id="z2",
        name="General operational reason",
        utility=0.70,
        privacy=0.05,
        cognitive_cost=0.15,
    ),
    mst_core.InformationItem(
        item_id="z3",
        name="Current system condition",
        utility=0.50,
        privacy=0.10,
        cognitive_cost=0.20,
    ),
]

clean_exact = mst_core.exact_mst(
    items=clean_test_items,
    tau=0.90,
)

clean_greedy = mst_core.greedy_mst(
    items=clean_test_items,
    tau=0.90,
)

print("Exact selected:", clean_exact["selected_items"])
print(f"Exact cost: {clean_exact['disclosure_cost']:.4f}")

print("Greedy selected:", clean_greedy["selected_items"])
print(f"Greedy cost: {clean_greedy['disclosure_cost']:.4f}")